In [18]:
from datasets import Dataset
import pandas as pd
import torch
from torch.optim import AdamW 
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, GenerationConfig, get_scheduler
from peft import LoraConfig, TaskType, get_peft_model

## 模型加载

In [2]:
model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [3]:
tokenizer.pad_token

'<|endoftext|>'

In [4]:
def process_func(example):
    MAX_LEN = 1024

    # 1. 构造消息序列（符合Qwen模板结构）
    messages = [
        {"role": "system", "content": "你是皇帝身边的女人——甄嬛。"},
        {"role": "user", "content": example["instruction"] + example.get("input", "")},
        {"role": "assistant", "content": example["output"]}
    ]

    # 2. 生成完整token序列
    full_input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,  # 不自动加assistant起始符
        return_tensors=None
    )

    # 3. 生成到assistant开始前的前缀（用于mask掉）
    prefix_input_ids = tokenizer.apply_chat_template(
        messages[:-1],   # 只到user为止
        tokenize=True,
        add_generation_prompt=True    # 会自动加 "<|im_start|>assistant\n"
    )

    # 4. mask掉system+user部分
    labels = full_input_ids.copy()
    prefix_len = len(prefix_input_ids)
    labels[:prefix_len] = [-100] * prefix_len

    # 5. 截断
    if len(full_input_ids) > MAX_LEN:
        full_input_ids = full_input_ids[:MAX_LEN]
        labels = labels[:MAX_LEN]

    # 6. attention mask
    attention_mask = [1] * len(full_input_ids)

    return {
        "input_ids": full_input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


In [5]:
import os
os.getcwd()

'C:\\Users\\hhm18\\Desktop\\course'

## 数据处理

In [6]:
df = pd.read_json("./state3/Huanhuan_chat/chat_data/huanhuan.json")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3729 entries, 0 to 3728
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  3729 non-null   object
 1   input        3729 non-null   object
 2   output       3729 non-null   object
dtypes: object(3)
memory usage: 87.5+ KB


In [8]:
df.head()

,instruction,input,output
0,小姐，别的秀女都在求中选，唯有咱们小姐想被撂牌子，菩萨一定记得真真儿的——,,嘘——都说许愿说破是不灵的。
1,这个温太医啊，也是古怪，谁不知太医不得皇命不能为皇族以外的人请脉诊病，他倒好，十天半月便往咱...,,你们俩话太多了，我该和温太医要一剂药，好好治治你们。
2,嬛妹妹，刚刚我去府上请脉，听甄伯母说你来这里进香了。,,出来走走，也是散心。
3,嬛妹妹，我虽是一介御医，俸禄微薄，可是我保证会一生一世对你好，疼爱你，保护你，永远事事以你为...,,实初哥哥这么说，就枉顾我们一直以来的兄妹情谊了，嬛儿没有哥哥，一直把你当作自己的亲哥哥一样看...
4,实初虽然唐突了妹妹，却是真心实意地希望妹妹不要去应选，这不仅仅是因为我心里一直把妹妹当成……...,,我们两家是世交，昔年恩义不过是父亲随手之劳，不必挂怀。


In [9]:
ds = Dataset.from_pandas(df)

tran_split = ds.train_test_split(test_size=0.01 ,seed=42,)
train_data = tran_split["train"]

temp = tran_split["test"].train_test_split(test_size=0.5, seed=42)
valid_data = temp["train"]
test_data = temp["test"]
print(train_data, valid_data, test_data)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3691
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 19
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 19
})


In [10]:
tokenized_train = train_data.map(process_func, remove_columns=ds.column_names)
tokenized_valid = valid_data.map(process_func, remove_columns=ds.column_names)
tokenized_test = test_data.map(process_func, remove_columns=ds.column_names)

Map:   0%|          | 0/3691 [00:00<?, ? examples/s]

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

## 设置 LoraConfig 以及 TrainingConfig

In [11]:
config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                        "gate_proj", "up_proj", "down_proj"],
        inference_mode=False, # 训练模式
        r=8, # Lora 秩
        lora_alpha=32, # Lora alaph，具体作用参见 Lora 原理
        lora_dropout=0.1# Dropout 比例
    )

In [12]:
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395


In [13]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [19]:
len(train_data)

3691

In [37]:
# config:
lr = 1e-4
num_epoch=3
per_device_train_batch_size=4
gradient_accumulation_steps=4
logging_steps=10
num_train_epochs=3
save_steps=100               
learning_rate=1e-4
eval_steps=100

In [38]:
optimizer = AdamW(model.parameters(), lr=lr)
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=0.1*num_epoch*len(train_data),
    num_training_steps=num_epoch*len(train_data),
)

In [41]:
args = TrainingArguments(
        output_dir="./output/qwen3_instruct_lora",
        learning_rate=lr,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        logging_steps=logging_steps,
        num_train_epochs=num_train_epochs,
        save_steps=save_steps,                 
        save_on_each_node=True,
        # gradient_checkpointing=True,
        logging_dir="../tf-logs/huanhuan/rus",       
        report_to="tensorboard",
        eval_strategy="steps",
        eval_steps=eval_steps,  
    )
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        optimizers=(optimizer,lr_scheduler),
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

In [42]:
trainer.train() # 开始训练 

Step,Training Loss,Validation Loss


KeyboardInterrupt: 